# AI-Powered Research Paper Assistant

### Intelligent Semantic Search and NLP-based Research Analysis

##  Developed by
**Rishabh Bhardwaj**

##  Project Objectives

This project aims to simplify research paper discovery and analysis using Natural Language Processing (NLP) and Deep Learning.

The system allows users to:

-  Perform Semantic Search over research papers
-  Generate concise summaries
-  Extract important keywords
-  Identify named entities using NER
-  Answer questions from research papers
-  Compare research papers based on content

## Technologies Used

- Python
- Hugging Face Transformers
- Sentence Transformers
- FAISS
- KeyBERT
- spaCy
- Pandas
- NumPy

In [100]:
import pandas as pd
import numpy as np

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from keybert import KeyBERT
import faiss
import spacy

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [102]:
from datasets import load_dataset

print("Loading dataset... Please wait.")

dataset = load_dataset(
    "CShorten/ML-ArXiv-Papers",
    split="train"
)

print("Dataset loaded successfully!")

Loading dataset... Please wait.
Dataset loaded successfully!


In [103]:
import pandas as pd

df = pd.DataFrame(dataset)

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (117592, 4)


,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 117592 entries, 0 to 117591
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Unnamed: 0.1  117592 non-null  int64  
 1   Unnamed: 0    112592 non-null  float64
 2   title         117592 non-null  str    
 3   abstract      117592 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 141.6 MB


# Data Cleaning

In [6]:
# Keep only useful columns
df = df[["title", "abstract"]]

# Remove duplicate papers
df = df.drop_duplicates()

# Remove rows with missing values
df = df.dropna()

# Reset indexing
df = df.reset_index(drop=True)

print(df.shape)
df.head()

(117569, 2)


,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


##  Prepare Combined Paper Text

In [8]:
df["paper_text"] = df["title"] + ". " + df["abstract"]

In [9]:
df["paper_text"] = (
    df["paper_text"]
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Semantic Embeddings using Sentence Transformers

##  Load Sentence Transformer Model

In [ ]:
from sentence_transformers import SentenceTransformer

print("Downloading model...")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("✅ Model loaded successfully!")

✅ Model loaded successfully!


##  Generate Semantic Embeddings

In [ ]:
embeddings = model.encode(
    df["paper_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/1838 [00:00<?, ?it/s]

##  Save Embeddings

In [ ]:
import numpy as np

np.save("data/paper_embeddings.npy", embeddings)

print("✅ Embeddings saved successfully!")

✅ Embeddings saved successfully!


##  Build FAISS Search Index

In [ ]:
import faiss

embeddings = embeddings.astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])

index.add(embeddings)

print("Total indexed papers:", index.ntotal)

Total indexed papers: 117569


##  Save FAISS Index

In [ ]:
faiss.write_index(index, "data/paper_index.faiss")

print("✅ FAISS index saved!")

✅ FAISS index saved!


In [ ]:
print(embeddings.shape)

(117569, 384)


##  AI Research Paper Summarization

In [14]:
from transformers import pipeline

print("Loading DistilBART...")

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6"
)

print("✅ DistilBART Loaded Successfully!")

Loading DistilBART...


pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

c:\Rishabh\ML\Research-Paper-Assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rbks0\.cache\huggingface\hub\models--sshleifer--distilbart-cnn-12-6. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


✅ DistilBART Loaded Successfully!


In [66]:
def summarize_paper(text):

    words = len(text.split())

    if words < 50:
        return text

    max_len = min(80, int(words * 0.6))
    min_len = max(20, int(words * 0.3))

    summary = summarizer(
        text,
        max_length=max_len,
        min_length=min_len,
        do_sample=False
    )

    return summary[0]["summary_text"]

In [75]:
paper = results.iloc[0]["abstract"]

summary = summarize_paper(paper)

print(summary)

 Computer-aided medical image analysis plays a significant role in assisting medical practitioners for expert clinical diagnosis and deciding the optimal treatment plan . At present, convolutional neural networks (CNN) are the preferred choice for medical images analysis . Rapid advancements in three-dimensional (3D) imaging systems and the


In [76]:
def semantic_search(query, top_k=5):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = df.iloc[indices[0]][[
        "title",
        "abstract",
        "paper_text"
    ]].copy()

    results["Similarity Score"] = distances[0]

    return results

In [77]:
results = semantic_search(
    "deep learning for medical image segmentation"
)

results

,title,abstract,paper_text,Similarity Score
88865,Medical Image Segmentation with 3D Convolution...,Computer-aided medical image analysis plays ...,Medical Image Segmentation with 3D Convolution...,0.509460
61136,Deep Learning Based Segmentation of Various Br...,Semantic segmentation of medical images with...,Deep Learning Based Segmentation of Various Br...,0.551600
78433,Catalyzing Clinical Diagnostic Pipelines Throu...,Deep learning has made a remarkable impact i...,Catalyzing Clinical Diagnostic Pipelines Throu...,0.563898
41901,Deep Semantic Segmentation of Natural and Medi...,The semantic image segmentation task consist...,Deep Semantic Segmentation of Natural and Medi...,0.600004
68340,U-Net and its variants for medical image segme...,U-net is an image segmentation technique dev...,U-Net and its variants for medical image segme...,0.608442


In [79]:
paper = results.iloc[0]

print("=" * 120)
print("TITLE")
print("=" * 120)
print(paper["title"])

print("\n")

print("=" * 120)
print("ABSTRACT")
print("=" * 120)
print(paper["abstract"])

print("\n")

print("=" * 120)
print("AI GENERATED SUMMARY")
print("=" * 120)
print(summarize_paper(paper["abstract"]))

TITLE
Medical Image Segmentation with 3D Convolutional Neural Networks: A
  Survey


ABSTRACT
  Computer-aided medical image analysis plays a significant role in assisting
medical practitioners for expert clinical diagnosis and deciding the optimal
treatment plan. At present, convolutional neural networks (CNN) are the
preferred choice for medical image analysis. In addition, with the rapid
advancements in three-dimensional (3D) imaging systems and the availability of
excellent hardware and software support to process large volumes of data, 3D
deep learning methods are gaining popularity in medical image analysis. Here,
we present an extensive review of the recently evolved 3D deep learning methods
in medical image segmentation. Furthermore, the research gaps and future
directions in 3D medical image segmentation are discussed.



AI GENERATED SUMMARY
 Computer-aided medical image analysis plays a significant role in assisting medical practitioners for expert clinical diagnosis and dec

##  Keyword Extraction using KeyBERT

In [80]:
from keybert import KeyBERT

print("Loading KeyBERT...")

kw_model = KeyBERT(model)

print("✅ KeyBERT Loaded Successfully!")

Loading KeyBERT...
✅ KeyBERT Loaded Successfully!


In [81]:
def extract_keywords(text, top_n=10):

    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1,3),
        stop_words="english",
        use_maxsum=True,
        nr_candidates=20,
        top_n=top_n
    )

    return keywords

In [82]:
keywords = extract_keywords(
    results.iloc[0]["paper_text"]
)

for word, score in keywords:
    print(f"{word:<35} {score:.4f}")

medical image                       0.4228
dimensional 3d imaging              0.4343
evolved 3d deep                     0.4486
volumes data 3d                     0.4566
directions 3d medical               0.4674
medical image analysis              0.4723
3d convolutional                    0.4965
medical image segmentation          0.5850
3d deep learning                    0.5855
segmentation 3d convolutional       0.6309


##  Named Entity Recognition (NER)

In [83]:
import spacy

nlp = spacy.load("en_core_web_sm")

print("✅ spaCy Loaded Successfully!")

✅ spaCy Loaded Successfully!


##  Research Entity Analyzer

In [84]:
def extract_entities(text):

    doc = nlp(text)

    entities = []

    for ent in doc.ents:
        entities.append((ent.text, ent.label_))

    return entities

In [85]:
entities = extract_entities(results.iloc[0]["paper_text"])

for entity, label in entities:
    print(f"{entity:<40} {label}")

3D Convolutional Neural Networks         ORG
CNN                                      ORG
three                                    CARDINAL
3D                                       ORG


In [87]:
research_models = [
    "CNN","RNN","LSTM","GRU",
    "Transformer","BERT","GPT",
    "ResNet","DenseNet","U-Net",
    "YOLO","ViT"
]

techniques = [
    "Deep Learning",
    "Machine Learning",
    "Segmentation",
    "Classification",
    "Object Detection",
    "Image Processing"
]

medical_terms = [
    "MRI","CT","X-ray",
    "Medical Image",
    "Tumor",
    "Brain"
]

text = results.iloc[0]["paper_text"].lower()

print("="*60)
print("RESEARCH ENTITY ANALYZER")
print("="*60)

print("\nModels:")
for item in research_models:
    if item.lower() in text:
        print("•", item)

print("\nTechniques:")
for item in techniques:
    if item.lower() in text:
        print("•", item)

print("\nMedical Terms:")
for item in medical_terms:
    if item.lower() in text:
        print("•", item)

print("\nGeneral Named Entities:")
doc = nlp(results.iloc[0]["paper_text"])

for ent in doc.ents:
    print(f"• {ent.text} ({ent.label_})")

RESEARCH ENTITY ANALYZER

Models:
• CNN

Techniques:
• Deep Learning
• Segmentation

Medical Terms:
• CT
• Medical Image

General Named Entities:
• 3D Convolutional Neural Networks (ORG)
• CNN (ORG)
• three (CARDINAL)
• 3D (ORG)


##  Similar Paper Recommendation

In [88]:
def recommend_similar_papers(selected_index, top_k=5):
    """
    Recommend papers similar to a selected paper.
    """

    paper_embedding = embeddings[selected_index].reshape(1, -1).astype("float32")

    distances, indices = index.search(paper_embedding, top_k + 1)

    # Skip the first result (it's the same paper)
    similar_indices = indices[0][1:]
    similarity_scores = distances[0][1:]

    recommendations = df.iloc[similar_indices][
        ["title", "abstract"]
    ].copy()

    recommendations["Similarity (%)"] = (1 / (1 + similarity_scores)) * 100

    return recommendations

In [89]:
paper_title = results.iloc[0]["title"]

selected_index = df[df["title"] == paper_title].index[0]

similar_papers = recommend_similar_papers(selected_index)

similar_papers

,title,abstract,Similarity (%)
52190,3D Deep Learning on Medical Images: A Review,"The rapid advancements in machine learning, ...",80.302628
46163,Evaluation of Multi-Slice Inputs to Convolutio...,When using Convolutional Neural Networks (CN...,68.332497
49369,Deep Learning in Medical Ultrasound Image Segm...,"Applying machine learning technologies, espe...",66.091278
78433,Catalyzing Clinical Diagnostic Pipelines Throu...,Deep learning has made a remarkable impact i...,65.805450
71769,Spatial Context-Aware Self-Attention Model For...,Multi-organ segmentation is one of most succ...,65.544243


In [90]:
print("="*120)
print("SIMILAR RESEARCH PAPERS")
print("="*120)

for i, row in similar_papers.iterrows():
    print("\nTitle:")
    print(row["title"])

    print("\nSimilarity Score:")
    print(f"{row['Similarity (%)']:.2f}%")

    print("-"*120)

SIMILAR RESEARCH PAPERS

Title:
3D Deep Learning on Medical Images: A Review

Similarity Score:
80.30%
------------------------------------------------------------------------------------------------------------------------

Title:
Evaluation of Multi-Slice Inputs to Convolutional Neural Networks for
  Medical Image Segmentation

Similarity Score:
68.33%
------------------------------------------------------------------------------------------------------------------------

Title:
Deep Learning in Medical Ultrasound Image Segmentation: a Review

Similarity Score:
66.09%
------------------------------------------------------------------------------------------------------------------------

Title:
Catalyzing Clinical Diagnostic Pipelines Through Volumetric Medical
  Image Segmentation Using Deep Neural Networks: Past, Present, & Future

Similarity Score:
65.81%
------------------------------------------------------------------------------------------------------------------------

Title

##  Paper Comparison

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def compare_papers(index1, index2):

    paper1 = df.iloc[index1]
    paper2 = df.iloc[index2]

    print("="*120)
    print("PAPER COMPARISON")
    print("="*120)

    print("\n📄 PAPER 1")
    print("-"*120)
    print(paper1["title"])

    print("\n📄 PAPER 2")
    print("-"*120)
    print(paper2["title"])

    # ---------------- SUMMARY ---------------- #

    print("\n" + "="*120)
    print("SUMMARY OF PAPER 1")
    print("="*120)
    print(summarize_paper(paper1["abstract"]))

    print("\n" + "="*120)
    print("SUMMARY OF PAPER 2")
    print("="*120)
    print(summarize_paper(paper2["abstract"]))

    # ---------------- KEYWORDS ---------------- #

    keywords1 = set()
    keywords2 = set()

    for keyword, score in extract_keywords(paper1["paper_text"]):
        for word in keyword.lower().replace("-", " ").split():
            keywords1.add(word)

    for keyword, score in extract_keywords(paper2["paper_text"]):
        for word in keyword.lower().replace("-", " ").split():
            keywords2.add(word)

    common = sorted(keywords1 & keywords2)
    unique1 = sorted(keywords1 - keywords2)
    unique2 = sorted(keywords2 - keywords1)

    # ---------------- SIMILARITY ---------------- #

    similarity = cosine_similarity(
        embeddings[index1].reshape(1,-1),
        embeddings[index2].reshape(1,-1)
    )[0][0]

    print("\n" + "="*120)
    print("COMPARISON REPORT")
    print("="*120)

    print(f"\n Cosine Similarity : {similarity*100:.2f}%")

    print("\nCommon Keywords")
    print("-"*60)

    if common:
        for word in common:
            print("•", word)
    else:
        print("No common keywords found.")

    print("\n Unique to Paper 1")
    print("-"*60)

    for word in unique1:
        print("•", word)

    print("\n Unique to Paper 2")
    print("-"*60)

    for word in unique2:
        print("•", word)

    # ---------------- AI CONCLUSION ---------------- #

    print("\n" + "="*120)
    print("AI CONCLUSION")
    print("="*120)

    if similarity >= 0.80:
        print("These papers discuss almost the same research topic with highly similar methodologies.")

    elif similarity >= 0.60:
        print("These papers belong to the same research area but focus on different aspects or applications.")

    elif similarity >= 0.40:
        print("These papers share a general research domain but solve different problems.")

    else:
        print("These papers are only loosely related.")

In [92]:
paper1_title = results.iloc[0]["title"]
paper2_title = results.iloc[1]["title"]

index1 = df[df["title"] == paper1_title].index[0]
index2 = df[df["title"] == paper2_title].index[0]

compare_papers(index1, index2)

PAPER COMPARISON

📄 PAPER 1
------------------------------------------------------------------------------------------------------------------------
Medical Image Segmentation with 3D Convolutional Neural Networks: A
  Survey

📄 PAPER 2
------------------------------------------------------------------------------------------------------------------------
Deep Learning Based Segmentation of Various Brain Lesions for
  Radiosurgery

SUMMARY OF PAPER 1
 Computer-aided medical image analysis plays a significant role in assisting medical practitioners for expert clinical diagnosis and deciding the optimal treatment plan . At present, convolutional neural networks (CNN) are the preferred choice for medical images analysis . Rapid advancements in three-dimensional (3D) imaging systems and the

SUMMARY OF PAPER 2
 Semantic segmentation of medical images with deep learning models is rapidly developing . We benchmarked state-of-the-art deep learning algorithms on our clinical stereotactic radio

In [104]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    embeddings[index1].reshape(1,-1),
    embeddings[index2].reshape(1,-1)
)

print("\n\n COSINE SIMILARITY")
print("-"*120)
print(f"{similarity[0][0]*100:.2f}%")



 COSINE SIMILARITY
------------------------------------------------------------------------------------------------------------------------
64.21%


In [98]:
from collections import Counter

def research_trends(results):

    keyword_counter = Counter()

    for _, row in results.iterrows():

        keywords = extract_keywords(row["paper_text"], top_n=5)

        for keyword, score in keywords:
            keyword_counter[keyword] += 1

    print("="*120)
    print("RESEARCH TREND ANALYSIS")
    print("="*120)

    print("\nTop Research Topics\n")

    for keyword, count in keyword_counter.most_common(10):
        print(f"{keyword:<45} Found in {count} paper(s)")

In [99]:
research_trends(results)

RESEARCH TREND ANALYSIS

Top Research Topics

medical image analysis                        Found in 3 paper(s)
medical image                                 Found in 1 paper(s)
evolved 3d deep                               Found in 1 paper(s)
volumes data 3d                               Found in 1 paper(s)
3d convolutional                              Found in 1 paper(s)
radiosurgery                                  Found in 1 paper(s)
learning segmentation algorithms              Found in 1 paper(s)
brain lesions radiosurgery                    Found in 1 paper(s)
deep learning models                          Found in 1 paper(s)
medical images deep                           Found in 1 paper(s)


## Final Features Implemented

- Semantic Search using FAISS
- AI Summarization using DistilBART
- Keyword Extraction using KeyBERT
- Named Entity Recognition using spaCy
- Research Entity Analysis
- Similar Paper Recommendation
- Paper Comparison
- Research Trend Analysis